In [17]:
%matplotlib inline
%load_ext autoreload
%autoreload 2
from imports import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### Load Configs

In [18]:
from config import main_config
from src.data_io import catalog
from src.utils.sessions import SESSION_COLUMNS

INCLUDE_EYE_TRACKING_SESSIONS = main_config.INCLUDE_EYE_TRACKING_SESSIONS

## Compiling data from all subjects

In [20]:
reject_subject = main_config["moca_rejection"]

In [21]:
subject_metadata = catalog.load("raw_session_metadata")
subjects_from_metadata = subject_metadata["subject_id"].unique()

# every session table compiled by 0.00 and 0.01, in name order so the joined table's row order is fixed
compiled_data_files = sorted(catalog.files("compiled_session", filename="*.parquet"))
subjects_from_data_file = [f.stem.split("_")[0] for f in compiled_data_files]

In [22]:
ucla = ["CG", "COH", "MBY", "DP", "FUR", "LBR", "MAR", "SMI", "PAM", "RW", "BBK", "BER", "DCAM", "ALE", "DMO", "AJL", "SKU"]
ucla_asmHC_EM = ["CB1", "JW", "KR", "MEG"]
ucla_asmHC_KP = ["AV", "BC", "BF", "EM", "ES", "GF", "GP", "JA", "MRM", "SY"]

case_western = ["RBA", "RDE", "SGA", "LHO", "RSH", "RZA", "SNO"]

stanford = subject_metadata.loc[
    subject_metadata["subject_id"].str.match(r"^P\d+$", na=False)  # Ensure it starts with 'P' followed by numbers
    & (subject_metadata["subject_id"].str.extract(r"P(\d+)")[0].astype(float) <= 24),  # Extract number and compare
    "subject_id",
].unique()
harvard = subject_metadata.loc[
    subject_metadata["subject_id"].str.match(r"^P\d+$", na=False)  # Ensure it starts with 'P' followed by numbers
    & (subject_metadata["subject_id"].str.extract(r"P(\d+)")[0].astype(float) > 24),  # Extract number and compare
    "subject_id",
].unique()

harvard_hc = subject_metadata.loc[subject_metadata["subject_id"].str.match(r"^HC\d+$", na=False), "subject_id"].unique()

# go through the subject_metadata and update the site
for subject_id in subject_metadata["subject_id"]:
    if subject_id in ucla or subject_id in ucla_asmHC_EM or subject_id in ucla_asmHC_KP:
        site = "UCLA"
    elif subject_id in case_western:
        site = "Case_Western"
    elif subject_id in stanford:
        site = "Stanford"
    elif subject_id in harvard or subject_id in harvard_hc:
        site = "Harvard"
    else:
        site = "Unknown"
    subject_metadata.loc[subject_metadata["subject_id"] == subject_id, "experiment_site"] = site

In [23]:
# remove rejected subjects from subjects_from_metadata and subjects_from_data_file
subjects_from_metadata = [s for s in subjects_from_metadata if s not in reject_subject]
subjects_from_data_file = [s for s in subjects_from_data_file if s not in reject_subject]

In [24]:
print(f"Metadata contains extra: {set(subjects_from_metadata) - set(subjects_from_data_file)}")
print(f"Data files contain extra: {set(subjects_from_data_file) - set(subjects_from_metadata)}")
print(len(set(subjects_from_data_file)), len(set(subjects_from_metadata)))

# Only assert if *all* subject IDs follow the pattern (none outside it)
if len(subjects_from_metadata) == len(subjects_from_metadata) and len(subjects_from_data_file) == len(subjects_from_data_file):
    assert len(set(subjects_from_data_file)) == len(set(subjects_from_metadata)), "Mismatch between metadata and data files"
else:
    pass

Metadata contains extra: set()
Data files contain extra: set()
74 74


In [25]:
# eye-tracking sessions (Perugini EM) are left out unless the config includes them
asmHC_EM_files = [f for f in compiled_data_files if f.stem.split("_")[0] in ucla_asmHC_EM]
session_files = [f for f in compiled_data_files if INCLUDE_EYE_TRACKING_SESSIONS or f not in asmHC_EM_files]
print(f"session tables: {len(compiled_data_files)}, eye-tracking: {len(asmHC_EM_files)}, joined: {len(session_files)}")

HC_PD_files: 137, asmHC_KP_files: 10, asmHC_EM_files: 4


In [26]:
aggregate_df = pd.concat([catalog.read(f) for f in session_files], ignore_index=True)

if not INCLUDE_EYE_TRACKING_SESSIONS:
    # remove rows from metadata corresponding to eye-tracking sessions
    eye_tracking_subjects = [f.stem.split("_")[0] for f in asmHC_EM_files]
    subject_metadata = subject_metadata[~subject_metadata["subject_id"].isin(eye_tracking_subjects)]

# replace empty strings with NaN
aggregate_df.replace("", np.nan, inplace=True)

In [28]:
aggregate_df["subject_id"].unique()

array(['DP', 'HC6', 'P33', 'DMO', 'P27', 'PAM', 'COH', 'P1', 'P11', 'HC5',
       'P10', 'BER', 'P25', 'P13', 'P19', 'CG', 'BBK', 'RZA', 'FUR',
       'LBR', 'P34', 'P29', 'RBA', 'P6', 'P23', 'DCAM', 'HC8', 'P32',
       'P31', 'P20', 'P2', 'P12', 'LHO', 'P30', 'MBY', 'RDE', 'P18',
       'SKU', 'AJL', 'P3', 'P24', 'HC7', 'ALE', 'HC9', 'P7', 'P4', 'SGA',
       'HC2', 'MAR', 'P9', 'P8', 'RSH', 'P16', 'HC13', 'P15', 'P26', 'RW',
       'HC1', 'SMI', 'P28', 'HC12', 'P22', 'P17', 'SNO', 'HC3', 'P14',
       'AV', 'BF', 'SY', 'BC', 'GF', 'EM', 'GP', 'ES', 'JA', 'MRM'],
      dtype=object)

In [29]:
aggregate_df = aggregate_df[SESSION_COLUMNS]

In [30]:
catalog.save(aggregate_df, "compiled_trials")
catalog.save(subject_metadata, "compiled_metadata")